# 03 — Publication-Quality Figures and Tables (Pairwise Comparisons)

This notebook produces the deliverables for the poster / manuscript,
organised around the **two pairwise comparisons** defined in notebook 02:

| Pair        | State A   | State B               | Status     |
|-------------|-----------|-----------------------|------------|
| activation  | inactive  | active                | primary    |
| ligand      | active    | active_ligand_bound   | secondary  |

For each pair we generate:

1. **MD-feature time-series** for the features relevant to that pair.
2. **MD-feature distribution** plots (violin + box).
3. **Side-by-side spectrograms** for each instrument — A on top, B on bottom.
4. **2 × 3 cross-instrument spectrogram grid** — robustness check across timbres.
5. **Piano-roll comparison** for the main instrument.

In addition we produce:

6. **PCA of feature space** across all three states (joint view).
7. **Composite poster figure** containing the two pairs side-by-side.
8. **Summary statistics tables** for MD features and MIDI parameters.
9. **Methods table** ready to paste into the manuscript.

All figures are saved in **PNG + PDF + SVG** at 600 dpi.


## 1. Mount Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Install dependencies


In [2]:
!pip -q install pandas numpy matplotlib scipy scikit-learn librosa soundfile pretty_midi


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 82.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.1 MB/s eta 0:00:00


## 3. Imports, paths, publication-style matplotlib


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
from scipy.io import wavfile
from scipy import signal
import librosa
import librosa.display
import pretty_midi
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = Path('/content/drive/MyDrive/GPCR_Sonification')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
AUDIO_DIR = PROJECT_DIR / 'outputs' / 'audio'
FIG_DIR = PROJECT_DIR / 'outputs' / 'figures'
TABLE_DIR = PROJECT_DIR / 'outputs' / 'tables'
for d in [FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':         10,
    'axes.titlesize':    11,
    'axes.titleweight':  'bold',
    'axes.labelsize':    10,
    'axes.linewidth':    0.8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'legend.frameon':    False,
    'figure.dpi':        110,
    'savefig.dpi':       600,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.05,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

STATE_COLORS = {
    'inactive':            '#0072B2',  # blue
    'active':              '#E69F00',  # orange
    'active_ligand_bound': '#009E73',  # green
}
STATE_ORDER = ['inactive', 'active', 'active_ligand_bound']
INSTRUMENTS = ['piano', 'violin', 'flute']
MAIN_INSTRUMENT = 'piano'

# Short, readable display names — `active_ligand_bound` is too long for
# axis ticks and legends, so we use a wrapped/abbreviated form there.
STATE_DISPLAY = {
    'inactive':            'inactive',
    'active':              'active',
    'active_ligand_bound': 'active + ligand',
}
# Two-line variant for narrow x-tick slots (violin plots, distributions)
STATE_DISPLAY_WRAP = {
    'inactive':            'inactive',
    'active':              'active',
    'active_ligand_bound': 'active +\nligand',
}

def savefig_multi(fig, base, formats=('png', 'pdf', 'svg')):
    out = []
    for fmt in formats:
        p = Path(str(base) + '.' + fmt)
        fig.savefig(p)
        out.append(p)
    return out


## 4. Load data


In [4]:
df = pd.read_csv(PROCESSED_DIR / 'all_features.csv')
df['state'] = pd.Categorical(df['state'], categories=STATE_ORDER, ordered=True)

mapping_tables = {}
for state in STATE_ORDER:
    f = PROCESSED_DIR / f'{state}_sonification_mapping.csv'
    if f.exists():
        mapping_tables[state] = pd.read_csv(f)

print('States in feature CSV:', df['state'].unique().tolist())
print('Mapping tables found  :', list(mapping_tables))


States in feature CSV: ['inactive', 'active', 'active_ligand_bound']
Mapping tables found  : ['inactive', 'active', 'active_ligand_bound']


## 5. Pairwise comparison configuration

Each pair has its own set of "relevant features". For the activation
pair we use the activation-related geometry markers; for the ligand
pair we use the ligand-related observables (plus a couple of broad
markers to provide context).


In [5]:
FEATURE_LABELS = {
    'RMSD_Ca_A':             ('Cα RMSD',                                'Å'),
    'TM3_TM6_distance_A':    ('TM3-TM6 intracellular distance',         'Å'),
    'NPxxY_RMSD_A':          ('NPxxY RMSD',                             'Å'),
    'DRY_ionic_lock_A':      ('DRY ionic-lock distance (R3.50-E6.30)',  'Å'),
    'Ligand_min_distance_A': ('Ligand-receptor minimum distance',       'Å'),
    'Ligand_contact_count':  ('Ligand-receptor contact count',          'atoms'),
}

PAIRS = {
    'activation': {
        'label':    'Activation: inactive vs active',
        'short':    'inactive -> active',
        'states':   ['inactive', 'active'],
        'features': ['TM3_TM6_distance_A', 'NPxxY_RMSD_A',
                     'DRY_ionic_lock_A', 'RMSD_Ca_A'],
    },
    'ligand': {
        'label':    'Ligand binding: active apo vs active + ligand',
        'short':    'active -> active + ligand',
        'states':   ['active', 'active_ligand_bound'],
        'features': ['Ligand_min_distance_A', 'Ligand_contact_count',
                     'NPxxY_RMSD_A', 'RMSD_Ca_A'],
    },
}

# Filter PAIRS down to ones that are actually present in the data
PAIRS = {
    k: v for k, v in PAIRS.items()
    if all(s in df['state'].cat.categories and s in df['state'].unique() for s in v['states'])
}
print('Active pairs for this run:', list(PAIRS))


Active pairs for this run: ['activation', 'ligand']


## 6. Pairwise MD feature time-series

For each pair we produce one figure with a small grid of subplots
(one per relevant feature), each overlaying the two states involved
in that pair. This is the figure that makes the case that the MD
features differ between the two states being compared.


In [ ]:
def plot_pair_features_timeseries(pair_name, pair_cfg):
    feats = [f for f in pair_cfg['features']
             if f in df.columns and not df[f].dropna().empty]
    if not feats:
        print(f'  {pair_name}: no features available')
        return
    ncols = 2
    nrows = int(np.ceil(len(feats) / ncols))
    # constrained_layout reserves room for the suptitle and per-subplot
    # titles/legends so nothing overlaps. A bit more vertical space per
    # subplot keeps the legend out of the data when loc='best' falls
    # back to a top-right corner.
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(9.0, 2.8 * nrows),
                             squeeze=False, constrained_layout=True)
    axes = axes.ravel()
    for i, feature in enumerate(feats):
        ax = axes[i]
        for state in pair_cfg['states']:
            g = df[df['state'] == state]
            if g[feature].dropna().empty:
                continue
            ax.plot(g['frame'], g[feature],
                    color=STATE_COLORS[state],
                    linewidth=1.1, alpha=0.9,
                    label=STATE_DISPLAY.get(state, state))
        label, unit = FEATURE_LABELS.get(feature, (feature, ''))
        ax.set_xlabel('Frame')
        ax.set_ylabel(f'{label} ({unit})' if unit else label)
        ax.set_title(label)
        # Solid frame + semi-opaque background so the legend never
        # obscures data points sitting beneath it.
        ax.legend(loc='best', fontsize=8, frameon=True,
                  framealpha=0.88, edgecolor='none')
    for j in range(len(feats), len(axes)):
        axes[j].axis('off')
    fig.suptitle(pair_cfg['label'], fontweight='bold')
    base = FIG_DIR / f'fig_pair_{pair_name}_features_timeseries'
    savefig_multi(fig, base)
    plt.show()
    print(f'  saved: {base.name}.[png,pdf,svg]')

for pair_name, pair_cfg in PAIRS.items():
    plot_pair_features_timeseries(pair_name, pair_cfg)


## 7. Pairwise MD feature distributions

Distribution plots (violin + box overlay) make the *separation* between
states quantitatively visible — overlapping violins mean the two states
are not well-distinguished by that feature.


In [ ]:
def plot_pair_features_distributions(pair_name, pair_cfg):
    # Filter features to ensure they exist in df.columns and have non-empty data
    # for *all* states in the current pair_cfg.
    feats = []
    for f in pair_cfg['features']:
        if f in df.columns:
            has_data_for_all_states = True
            for s in pair_cfg['states']:
                if df.loc[df['state'] == s, f].dropna().empty:
                    has_data_for_all_states = False
                    break
            if has_data_for_all_states:
                feats.append(f)

    if not feats:
        print(f'  {pair_name}: no features with data for all states available for plotting')
        return

    # Wider per-feature slot + constrained_layout so the wrapped
    # 'active +\nligand' tick label, the y-axis label, and the suptitle
    # all fit without overlapping.
    fig, axes = plt.subplots(1, len(feats),
                             figsize=(3.1 * len(feats), 3.4),
                             constrained_layout=True)
    if len(feats) == 1:
        axes = [axes]
    for ax, feature in zip(axes, feats):
        data = [df.loc[df['state'] == s, feature].dropna().values
                for s in pair_cfg['states']]
        parts = ax.violinplot(data, showmeans=False, showmedians=False,
                              showextrema=False, widths=0.8)
        for j, pc in enumerate(parts['bodies']):
            pc.set_facecolor(STATE_COLORS[pair_cfg['states'][j]])
            pc.set_alpha(0.55)
            pc.set_edgecolor('black')
            pc.set_linewidth(0.5)
        ax.boxplot(data, widths=0.18, showfliers=False,
                   medianprops=dict(color='black', linewidth=1.2),
                   boxprops=dict(linewidth=0.7),
                   whiskerprops=dict(linewidth=0.7),
                   capprops=dict(linewidth=0.7))
        ax.set_xticks(range(1, len(pair_cfg['states']) + 1))
        # Two-line label for the long state name so we don't need
        # rotation (rotated labels were colliding with the axis line).
        ax.set_xticklabels(
            [STATE_DISPLAY_WRAP.get(s, s) for s in pair_cfg['states']],
            rotation=0,
        )
        label, unit = FEATURE_LABELS.get(feature, (feature, ''))
        ax.set_ylabel(f'{label} ({unit})' if unit else label)
        ax.set_title(label)
    fig.suptitle(pair_cfg['label'], fontweight='bold')
    base = FIG_DIR / f'fig_pair_{pair_name}_features_distributions'
    savefig_multi(fig, base)
    plt.show()
    print(f'  saved: {base.name}.[png,pdf,svg]')

for pair_name, pair_cfg in PAIRS.items():
    plot_pair_features_distributions(pair_name, pair_cfg)


## 8. Side-by-side spectrograms (one figure per pair × instrument)

For each (pair, instrument) we plot the spectrogram of state A on top
and state B on the bottom of the same figure, with the same colour
scale. This is the central audio-side claim: with the same instrument
and same mapping, the two states have visibly different spectro-temporal
content.


In [ ]:
def load_wav(wav_path):
    if not wav_path.exists():
        return None, None
    y, sr = librosa.load(str(wav_path), sr=None, mono=True)
    return y, sr

def stft_db(y, n_fft=2048, hop_length=512):
    return librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=n_fft,
                                                       hop_length=hop_length)),
                                   ref=np.max)

# Inter-note gap used by notebook 02 when laying out MIDI notes in
# real time. Audio_time(note i) = sum(duration[:i]) + i * NOTE_GAP_S.
NOTE_GAP_S = 0.02

def _note_start_times(mapping_df, gap=NOTE_GAP_S):
    '''Reconstruct the real audio start-time of each note in the
    mapping table (must match the layout used in notebook 02).'''
    durations = mapping_df['duration_s'].values
    # start[0] = 0; start[i] = start[i-1] + duration[i-1] + gap
    return np.concatenate([[0.0],
                           np.cumsum(durations + gap)[:-1]])

def add_md_frame_axis(ax, mapping_df, label='MD note index (sonification step)'):
    '''Attach a secondary x-axis at the top of `ax` that shows the
    sonification step (note index 0..N-1) corresponding to each audio
    time. Two spectrograms from different states with different total
    durations will then share the SAME top-axis range, so visual
    comparison "what's happening at step k in both states?" becomes
    direct even though the bottom (audio time) axes differ.'''
    if mapping_df is None or mapping_df.empty:
        return None
    starts = _note_start_times(mapping_df)
    n = len(starts)
    def t_to_idx(t):
        return np.interp(t, starts, np.arange(n))
    def idx_to_t(i):
        return np.interp(i, np.arange(n), starts)
    sec = ax.secondary_xaxis('top', functions=(t_to_idx, idx_to_t))
    sec.set_xlabel(label, fontsize=8, labelpad=2)
    sec.tick_params(axis='x', labelsize=7)
    return sec

def plot_pair_spectrograms_one_instrument(pair_name, pair_cfg, instrument):
    wavs   = [AUDIO_DIR / f'{s}_{instrument}.wav' for s in pair_cfg['states']]
    titles = [STATE_DISPLAY.get(s, s) for s in pair_cfg['states']]
    if not all(w.exists() for w in wavs):
        print(f'  {pair_name}/{instrument}: missing WAV(s)')
        return
    # NOTE: do NOT use sharex=True. The two states have different
    # audio durations (audio length itself encodes the NPxxY-RMSD
    # feature via per-note duration), so a shared x-axis would stretch
    # the shorter panel and leave it floating in white space. Each
    # panel renders edge-to-edge on its OWN audio-time axis (bottom),
    # and we add a SECONDARY axis on top that maps each panel's audio
    # time back to the common MD-note index (0..N-1). The top axis is
    # the same across panels — read it to compare "what happens at
    # step k" between states.
    fig, axes = plt.subplots(2, 1, figsize=(8.5, 5.8),
                             constrained_layout=True)
    img_last = None
    for ax, w, ttl, state in zip(axes, wavs, titles, pair_cfg['states']):
        y, sr = load_wav(w)
        D = stft_db(y)
        img_last = librosa.display.specshow(D, sr=sr, x_axis='time',
                                            y_axis='log', cmap='magma', ax=ax)
        ax.set_title(ttl, color=STATE_COLORS[state], pad=22)
        # Secondary top axis: MD-note index (state-comparable)
        add_md_frame_axis(ax, mapping_tables.get(state))
    # Suppress the word "Time" on the top panel so the next panel's
    # title sits cleanly under it (tick numbers stay).
    axes[0].set_xlabel('')
    fig.colorbar(img_last, ax=axes, format='%+2.0f dB', pad=0.02)
    fig.suptitle(f'{pair_cfg["label"]} — {instrument}', fontweight='bold')
    base = FIG_DIR / f'fig_pair_{pair_name}_spectrograms_{instrument}'
    savefig_multi(fig, base)
    plt.show()
    print(f'  saved: {base.name}.[png,pdf,svg]')

for pair_name, pair_cfg in PAIRS.items():
    for instrument in INSTRUMENTS:
        plot_pair_spectrograms_one_instrument(pair_name, pair_cfg, instrument)


## 9. Cross-instrument robustness grid (one figure per pair)

A 2 × 3 grid of spectrograms: rows = the two states of the pair,
columns = piano / violin / flute. If the auditory distinction between
the two states truly comes from the MD and not from the instrument, the
within-column contrast (top vs bottom) should be visible in every
column.


In [ ]:
def plot_pair_cross_instrument_grid(pair_name, pair_cfg):
    states = pair_cfg['states']
    n_rows = len(states)
    n_cols = len(INSTRUMENTS)
    # sharex='row': within a row all 3 instrument panels show the SAME
    # state (so they have identical duration), but rows correspond to
    # different states with different durations — sharing globally
    # would stretch the shorter state's panels into empty space.
    # sharey=True: every spectrogram has the same log-frequency axis
    # (0 .. sr/2), so it's safe and reduces visual noise to share it.
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(3.8 * n_cols, 3.1 * n_rows),
                             squeeze=False, sharex='row', sharey=True,
                             constrained_layout=True)
    img_last = None
    for i, state in enumerate(states):
        for j, instrument in enumerate(INSTRUMENTS):
            ax = axes[i, j]
            w = AUDIO_DIR / f'{state}_{instrument}.wav'
            if not w.exists():
                ax.axis('off')
                continue
            y, sr = load_wav(w)
            D = stft_db(y)
            img_last = librosa.display.specshow(D, sr=sr, x_axis='time',
                                                y_axis='log',
                                                cmap='magma', ax=ax)
            # Column title (instrument) only on the top row;
            # row label (state) only on the left column.
            if i == 0:
                ax.set_title(instrument, fontweight='bold', pad=22)
            else:
                ax.set_title('')
            if j == 0:
                ax.set_ylabel(STATE_DISPLAY.get(state, state),
                              color=STATE_COLORS[state],
                              fontweight='bold')
            else:
                ax.set_ylabel('')
            # Drop the "Time" word on the top row (tick numbers remain)
            if i < n_rows - 1:
                ax.set_xlabel('')
            # Top secondary axis: MD-note index, only on the topmost
            # panel in each row (sharex='row' would propagate but
            # secondary axes don't propagate via sharex, so we add it
            # explicitly only where the instrument title lives — that
            # keeps the rest of the row uncluttered while still showing
            # the comparable note-index scale once per state.)
            if i == 0 and j == 0:
                add_md_frame_axis(ax, mapping_tables.get(state),
                                  label=f'{STATE_DISPLAY.get(state, state)} '
                                        f'— note index')
            if i == 1 and j == 0:
                add_md_frame_axis(ax, mapping_tables.get(state),
                                  label=f'{STATE_DISPLAY.get(state, state)} '
                                        f'— note index')
    if img_last is not None:
        fig.colorbar(img_last, ax=axes, format='%+2.0f dB', pad=0.02)
    fig.suptitle(f'{pair_cfg["label"]} — cross-instrument robustness',
                 fontweight='bold')
    base = FIG_DIR / f'fig_pair_{pair_name}_cross_instrument_grid'
    savefig_multi(fig, base)
    plt.show()
    print(f'  saved: {base.name}.[png,pdf,svg]')

for pair_name, pair_cfg in PAIRS.items():
    plot_pair_cross_instrument_grid(pair_name, pair_cfg)


## 10. Piano-roll comparison per pair

Visual analogue of the audio: each note is a coloured bar (state-coloured,
velocity-shaded). Two stacked panels per pair, one for each state.


In [ ]:
def plot_piano_roll_into(ax, mapping_df, color):
    t = 0.0
    starts, ends, pitches, vels = [], [], [], []
    for _, row in mapping_df.iterrows():
        s = t
        e = t + float(row['duration_s'])
        starts.append(s); ends.append(e)
        pitches.append(int(row['midi_note']))
        vels.append(int(row['velocity']))
        t = e + 0.02
    vels = np.array(vels)
    alphas = 0.35 + 0.6 * (vels - vels.min()) / max(np.ptp(vels), 1)
    for s, e, p, a in zip(starts, ends, pitches, alphas):
        ax.broken_barh([(s, e - s)], (p - 0.4, 0.8),
                       facecolors=color, alpha=a, edgecolor='none')
    ax.set_ylabel('MIDI note')
    ax.set_xlim(0, ends[-1] if ends else 1)
    if pitches:
        ax.set_ylim(min(pitches) - 2, max(pitches) + 2)

def plot_pair_piano_roll(pair_name, pair_cfg):
    states = pair_cfg['states']
    if not all(s in mapping_tables for s in states):
        return
    # constrained_layout so the suptitle doesn't push into the top
    # panel title and the bottom panel x-label doesn't clip.
    fig, axes = plt.subplots(len(states), 1,
                             figsize=(9.0, 2.4 * len(states)),
                             sharex=False, constrained_layout=True)
    if len(states) == 1:
        axes = [axes]
    for ax, state in zip(axes, states):
        plot_piano_roll_into(ax, mapping_tables[state], STATE_COLORS[state])
        ax.set_title(STATE_DISPLAY.get(state, state),
                     color=STATE_COLORS[state])
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'{pair_cfg["label"]} — piano-roll', fontweight='bold')
    base = FIG_DIR / f'fig_pair_{pair_name}_piano_roll'
    savefig_multi(fig, base)
    plt.show()
    print(f'  saved: {base.name}.[png,pdf,svg]')

for pair_name, pair_cfg in PAIRS.items():
    plot_pair_piano_roll(pair_name, pair_cfg)


## 11. PCA of feature space (joint view of all three states)

The pairwise figures focus on two-state contrasts. A complementary
joint view is provided by PCA over all three states; if the chosen
features encode activation correctly, the three states should appear
as distinguishable clouds in PC1-PC2 space.


In [ ]:
# Use features common to both pairs (drop ligand-only features so all states qualify)
joint_features = ['TM3_TM6_distance_A', 'NPxxY_RMSD_A',
                  'DRY_ionic_lock_A', 'RMSD_Ca_A']
joint_features = [f for f in joint_features
                  if f in df.columns and not df[f].dropna().empty]
sub = df[['state'] + joint_features].dropna()
X = StandardScaler().fit_transform(sub[joint_features].values)
pca = PCA(n_components=2)
Z = pca.fit_transform(X)
sub = sub.assign(PC1=Z[:, 0], PC2=Z[:, 1])

fig, ax = plt.subplots(figsize=(5.4, 4.6), constrained_layout=True)
for state in STATE_ORDER:
    s = sub[sub['state'] == state]
    if s.empty:
        continue
    ax.scatter(s['PC1'], s['PC2'], s=10, alpha=0.55,
               edgecolor='none', color=STATE_COLORS[state],
               label=STATE_DISPLAY.get(state, state))
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f} %)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f} %)')
ax.set_title('PCA of MD feature space (all states)')
# Solid frame so the legend never disappears against dense scatter
# clouds underneath; explicit upper-right anchor avoids `loc='best'`
# silently choosing a spot that overlaps a state cluster.
ax.legend(loc='upper right', frameon=True, framealpha=0.9,
          edgecolor='none')
base = FIG_DIR / 'fig_feature_pca'
savefig_multi(fig, base)
plt.show()

loadings = pd.DataFrame(pca.components_.T, index=joint_features, columns=['PC1', 'PC2'])
loadings.to_csv(TABLE_DIR / 'pca_loadings.csv')
display(loadings)


## 12. Summary statistics tables

Two tables for the supplementary material:

- `feature_summary_statistics.csv` — mean / std / min / max per state for the MD features.
- `midi_mapping_summary_statistics.csv` — the same for the resolved MIDI parameters.


In [14]:
all_used_features = sorted({f for p in PAIRS.values() for f in p['features']
                            if f in df.columns and not df[f].dropna().empty})
summary = df.groupby('state', observed=True)[all_used_features].agg(['mean', 'std', 'min', 'max'])
summary.to_csv(TABLE_DIR / 'feature_summary_statistics.csv')
display(summary)
print('Saved:', TABLE_DIR / 'feature_summary_statistics.csv')

if mapping_tables:
    map_df = pd.concat(mapping_tables.values(), ignore_index=True)
    midi_summary = (map_df.groupby('state', observed=True)
                          .agg({'midi_note':  ['mean', 'std', 'min', 'max'],
                                'duration_s': ['mean', 'std', 'min', 'max'],
                                'velocity':   ['mean', 'std', 'min', 'max']}))
    midi_summary.to_csv(TABLE_DIR / 'midi_mapping_summary_statistics.csv')
    display(midi_summary)
    print('Saved:', TABLE_DIR / 'midi_mapping_summary_statistics.csv')


DRY_ionic_lock_A                                 \
                                mean       std       min        max   
state                                                                 
inactive                    8.490786  1.218391  5.706927  11.817753   
active                     10.194207  1.591909  7.855375  14.932412   
active_ligand_bound        13.046817  1.685953  8.429572  16.544834   

                    Ligand_contact_count                           \
                                    mean        std    min    max   
state                                                               
inactive                             NaN        NaN    NaN    NaN   
active                               NaN        NaN    NaN    NaN   
active_ligand_bound              188.498  29.132858  108.0  254.0   

                    Ligand_min_distance_A            ... NPxxY_RMSD_A  \
                                     mean       std  ...          min   
state                                                ...                
inactive                              NaN       NaN  ...          0.0   
active                                NaN       NaN  ...          0.0   
active_ligand_bound              1.805607  0.169431  ...          0.0   

                              RMSD_Ca_A                           \
                          max      mean       std  min       max   
state                                                              
inactive             4.870252  3.441642  0.776114  0.0  5.371986   
active               6.917078  4.062474  0.896646  0.0  5.268927   
active_ligand_bound  3.421754  4.795811  1.134218  0.0  7.406072   

                    TM3_TM6_distance_A                                  
                                  mean       std        min        max  
state                                                                   
inactive                     11.855666  1.303755   8.897036  14.956400  
active                       13.802352  1.535996  11.327837  18.479071  
active_ligand_bound          16.170442  1.946286   9.714166  19.927354  

[3 rows x 24 columns]

Saved: /content/drive/MyDrive/GPCR_Sonification/outputs/tables/feature_summary_statistics.csv


midi_note                   duration_s                  \
                         mean       std min max       mean       std   min   
state                                                                        
active                 64.022  5.093663  55  79   0.314877  0.069256  0.12   
active_ligand_bound    71.746  6.368249  50  84   0.219381  0.032610  0.12   
inactive               57.726  4.324851  48  67   0.213740  0.037887  0.12   

                              velocity                      
                          max     mean        std min  max  
state                                                       
active               0.450000   77.912   8.482226  40   89  
active_ligand_bound  0.283245   84.822  10.718025  40  110  
inactive             0.352350   72.030   7.337355  40   90

Saved: /content/drive/MyDrive/GPCR_Sonification/outputs/tables/midi_mapping_summary_statistics.csv


## 13. Methods table


In [15]:
methods_rows = [
    ('Study design',
     'Two pairwise comparisons under an identical sonification mapping: '
     '(1) activation pair = inactive vs active (apo), '
     '(2) ligand pair = active apo vs active + bound agonist. '
     'Each pair is rendered with three instruments (piano, violin, flute) '
     'to test whether the auditory state distinction is robust to timbre.'),
    ('Data source',
     'β2AR MD trajectories were obtained from GPCRMD: '
     'inactive (Dynamic ID 11), active apo (Dynamic ID 116), '
     'active with bound orthosteric agonist (Dynamic ID 117).'),
    ('Trajectory pre-processing',
     'Each trajectory was rigid-body aligned in memory on all Cα atoms using '
     'MDAnalysis.analysis.align.AlignTraj before feature extraction, so that '
     'downstream RMSDs reflect genuine internal conformational change.'),
    ('MD features',
     'Per frame: global Cα RMSD, TM3-TM6 intracellular Cα-centre distance, '
     'NPxxY motif RMSD, DRY ionic-lock distance (R3.50-E6.30), '
     'ligand-receptor minimum heavy-atom distance, and ligand-receptor '
     'contact count (≤4.5 Å).'),
    ('Normalisation',
     'Features were min-max normalised against the global distribution '
     'across all three states so that identical MD values map to identical '
     'musical values in every render.'),
    ('Sonification mapping',
     'TM3-TM6 distance → pitch (C-major pentatonic, MIDI 48-84); '
     'NPxxY RMSD → note duration (0.12-0.45 s); '
     'Cα RMSD → velocity (MIDI 40-110); '
     'DRY ionic-lock distance → sustained harmony pad intensity; '
     'ligand contact count → snare-percussion accents.'),
    ('Audio rendering',
     'MIDI scores written with pretty_midi, synthesised to 44.1 kHz WAV '
     'with FluidSynth using a General-MIDI SoundFont. Piano (GM 0) is the '
     'primary instrument; violin (40) and flute (73) are alternative '
     'timbre renderings for the robustness check.'),
    ('Outputs',
     'Per-state per-instrument MIDI/WAV (9 files); pairwise concatenation '
     'MIDI/WAV per instrument per pair (6 files); feature time-series, '
     'distributions, side-by-side spectrograms, cross-instrument grids, '
     'piano-roll comparisons, PCA, and summary tables.'),
]
methods_df = pd.DataFrame(methods_rows, columns=['Step', 'Description'])
methods_df.to_csv(TABLE_DIR / 'methods_table.csv', index=False)
display(methods_df)
print('Saved:', TABLE_DIR / 'methods_table.csv')


,Step,Description
0,Study design,Two pairwise comparisons under an identical so...
1,Data source,β2AR MD trajectories were obtained from GPCRMD...
2,Trajectory pre-processing,Each trajectory was rigid-body aligned in memo...
3,MD features,"Per frame: global Cα RMSD, TM3-TM6 intracellul..."
4,Normalisation,Features were min-max normalised against the g...
5,Sonification mapping,"TM3-TM6 distance → pitch (C-major pentatonic, ..."
6,Audio rendering,"MIDI scores written with pretty_midi, synthesi..."
7,Outputs,Per-state per-instrument MIDI/WAV (9 files); p...


Saved: /content/drive/MyDrive/GPCR_Sonification/outputs/tables/methods_table.csv


## 14. Composite poster figure

A single multi-panel figure designed to fit on a conference poster.
The top row covers the activation pair (primary claim); the bottom row
covers the ligand pair (secondary claim). For each pair we show, left
to right: the key feature time-series, a side-by-side piano
spectrogram, and a small instrument-robustness strip.


In [ ]:
def add_spectrogram_panel(ax, wav_path, title, ylabel=None, color=None):
    if not wav_path.exists():
        ax.axis('off')
        return None
    y, sr = librosa.load(str(wav_path), sr=None, mono=True)
    D = stft_db(y)
    img = librosa.display.specshow(D, sr=sr, x_axis='time',
                                   y_axis='log', cmap='magma', ax=ax)
    if title:
        ax.set_title(title, color=color or 'black')
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    return img

def add_feature_panel(ax, pair_cfg, feature):
    label, unit = FEATURE_LABELS.get(feature, (feature, ''))
    for state in pair_cfg['states']:
        g = df[df['state'] == state]
        if g[feature].dropna().empty:
            continue
        ax.plot(g['frame'], g[feature],
                color=STATE_COLORS[state], linewidth=1.0,
                alpha=0.9, label=STATE_DISPLAY.get(state, state))
    ax.set_xlabel('Frame')
    ax.set_ylabel(f'{label} ({unit})' if unit else label)
    ax.set_title(label)
    ax.legend(loc='best', fontsize=8, frameon=True,
              framealpha=0.88, edgecolor='none')

# Pick the "headline feature" for each pair
HEADLINE_FEATURE = {
    'activation': 'TM3_TM6_distance_A',
    'ligand':     'Ligand_contact_count',
}

# Wider canvas + constrained_layout. With GridSpec we must hand the
# layout engine the GridSpec instance so it can negotiate column widths
# against the panel that spans both rows (the PCA column).
fig = plt.figure(figsize=(14.0, 9.6), constrained_layout=True)
gs = GridSpec(2, 4, figure=fig,
              width_ratios=[1.4, 1.4, 1.4, 1.4])

for row, (pair_name, pair_cfg) in enumerate(PAIRS.items()):
    state_a, state_b = pair_cfg['states']

    # Col 0: headline feature time-series
    axF = fig.add_subplot(gs[row, 0])
    add_feature_panel(axF, pair_cfg, HEADLINE_FEATURE.get(pair_name, pair_cfg['features'][0]))
    axF.set_title(f'{chr(65 + row*4)} | {axF.get_title()}', loc='left')

    # Col 1: spectrogram state A (piano)
    axA = fig.add_subplot(gs[row, 1])
    add_spectrogram_panel(axA, AUDIO_DIR / f'{state_a}_{MAIN_INSTRUMENT}.wav',
                          title=f'{STATE_DISPLAY.get(state_a, state_a)} (piano)',
                          color=STATE_COLORS[state_a])
    axA.set_title(f'{chr(65 + row*4 + 1)} | {axA.get_title()}',
                  loc='left', color=STATE_COLORS[state_a])

    # Col 2: spectrogram state B (piano)
    axB = fig.add_subplot(gs[row, 2])
    add_spectrogram_panel(axB, AUDIO_DIR / f'{state_b}_{MAIN_INSTRUMENT}.wav',
                          title=f'{STATE_DISPLAY.get(state_b, state_b)} (piano)',
                          color=STATE_COLORS[state_b])
    axB.set_title(f'{chr(65 + row*4 + 2)} | {axB.get_title()}',
                  loc='left', color=STATE_COLORS[state_b])

    # Col 3: PCA on the right, spanning both rows. Only set up once.
    if row == 0:
        axP = fig.add_subplot(gs[:, 3])
        for state in STATE_ORDER:
            s = sub[sub['state'] == state]
            if s.empty:
                continue
            axP.scatter(s['PC1'], s['PC2'], s=12, alpha=0.55,
                        edgecolor='none', color=STATE_COLORS[state],
                        label=STATE_DISPLAY.get(state, state))
        axP.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f} %)')
        axP.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f} %)')
        axP.set_title('D | PCA of MD feature space', loc='left')
        axP.legend(loc='upper right', fontsize=8, frameon=True,
                   framealpha=0.9, edgecolor='none')

fig.suptitle('β2AR pairwise sonification: activation and ligand-binding '
             'auditory signatures',
             fontsize=13, fontweight='bold')
base = FIG_DIR / 'fig_composite_poster'
savefig_multi(fig, base)
plt.show()
print('Saved:', base.with_suffix('.png').name)


## 15. Suggested caption / interpretation

For the manuscript and the oral presentation, suggested framing:

> Under an identical sonification mapping and identical instrument,
> the β2AR active and inactive trajectories produce visibly and
> audibly distinct dynamic signatures (activation pair). The same
> mapping applied to the active apo and active ligand-bound
> trajectories yields a separate, less pronounced but still
> discriminable distinction (ligand pair). The fact that both pairwise
> contrasts persist across three different instruments (piano, violin,
> flute) indicates that the auditory state separation reflects the
> underlying MD differences rather than instrument-specific timbre
> artefacts. Sonification is presented here as a complementary
> perceptual layer to classical structural analysis, not as a
> replacement for it.

### Output of this notebook

```text
outputs/figures/fig_pair_activation_features_timeseries.{png,pdf,svg}
outputs/figures/fig_pair_activation_features_distributions.{png,pdf,svg}
outputs/figures/fig_pair_activation_spectrograms_{piano,violin,flute}.{png,pdf,svg}
outputs/figures/fig_pair_activation_cross_instrument_grid.{png,pdf,svg}
outputs/figures/fig_pair_activation_piano_roll.{png,pdf,svg}

outputs/figures/fig_pair_ligand_features_timeseries.{png,pdf,svg}
outputs/figures/fig_pair_ligand_features_distributions.{png,pdf,svg}
outputs/figures/fig_pair_ligand_spectrograms_{piano,violin,flute}.{png,pdf,svg}
outputs/figures/fig_pair_ligand_cross_instrument_grid.{png,pdf,svg}
outputs/figures/fig_pair_ligand_piano_roll.{png,pdf,svg}

outputs/figures/fig_feature_pca.{png,pdf,svg}
outputs/figures/fig_composite_poster.{png,pdf,svg}

outputs/tables/feature_summary_statistics.csv
outputs/tables/midi_mapping_summary_statistics.csv
outputs/tables/pca_loadings.csv
outputs/tables/methods_table.csv
```
